# 训推不一致，有哪些现成解决方案

目前训推不一致（Training-Inference Mismatch）已经有几类比较成熟的现成方案，可以按“**框架级对齐** → **算子/精度级对齐** → **MoE 路由对齐** → **算法层兜底**”来选。

## 框架级对齐方案

| 方案 | 核心做法 | 适用场景 |
|---|---|---|
| **vLLM + TorchTitan bitwise on-policy** | 让训练和推理复用同一套前向 kernel，并为推理算子补写反向，实现训练/推理 bitwise 一致。 | 想从底层 kernel 层彻底对齐，且能接受一定性能开销 |
| **SGLang 确定性推理 / Training-Serving Alignment** | 通过确定性推理、权重 refit、rollout 控制等方式，降低 rollout 与训练之间的数值差异。 | 使用 SGLang 做 rollout，配合 Megatron/Torch 训练 |
| **slime 训推对齐方案** | 在 Megatron 训练 + SGLang rollout 链路中，对齐 FP8、KV Cache、DeepEP、DeepGEMM、稀疏注意力等路径。 | **大规模 RL 后训练，尤其是 GLM 类链路** |
| **verl 训练侧重算 logprob** | <font color='red'>rollout 侧生成的 logprob 不直接用于训练梯度，训练侧用训练引擎重算 old logprob。</font> | **PPO/GRPO 等 RLHF 框架通用做法** |
| **昇腾训推一致算子集** | 基于 Ascend C 约束关键算子，实现注意力语义、reduce 序、softmax 分块、精度路径统一。 | 昇腾硬件栈，或想参考其算子约束思路 |

## 算子与精度对齐
这类方案最接近“底层根治”。
  - 统一 Attention 语义：
    - 训练 mask softmax、推理 prefill/decode 的可见 token 集合要对齐。
  - 锁定 reduce 累加序：
    - 训练 FA、推理 PFA/PagedAttention 在规约轴上的切分顺序要一致。
  - 对齐在线 softmax 分块：
    - 训练和推理的 softmax 分块大小、归约节奏要一致。
  - 统一关键路径精度：
    - softmax 累加、KV cache、router logits、LM head 等敏感位置尽量用相同精度。
  - FP16 替代 BF16：
    - BF16 尾数较少，RL 中容易放大训推差异；统一 FP16 可明显改善稳定性，但要注意动态范围和 loss scaling。

## MoE 路由回放
<font color='red'>MoE 模型里，推理和训练即使权重相同，也可能因为 router logits 精度、并行切分、top-k 实现不同而选出不同专家。</font>
  - ROLL Router Replay：<font color='blue'>训练时复用 rollout 阶段记录的专家路由 mask，而不是重新做 top-k。</font>
  - R2：训练内部 old policy forward 记录路由，后续梯度更新中复放，避免路由随权重漂移。
  - R3：直接从 SGLang/vLLM rollout 中取路由 mask，训练侧复放，同时缓解训推差异和策略 stale。
  
如果用的是 MoE，Router Replay 通常比单纯调精度收益更明显。

## 算法层兜底
如果暂时做不到完全数值对齐，可以用算法手段缓解：
  - 重要性采样截断/掩码：限制 rollout 分布与训练分布差异带来的权重爆炸。
  - KL 惩罚：抑制策略偏离参考模型太远。
  - Clipping：PPO/GRPO 中限制新旧策略比率。
  - 重算 old logprob：不直接信任 rollout 返回的 logprob，训练侧用训练引擎重新前向。
  
<font color='red'>但这些更适合“残余 diff”的兜底，不能替代底层算子和精度对齐。</font>

### 选型建议
  - 稠密模型 + vLLM rollout：优先看 vLLM batch-invariant / bitwise on-policy 方案，配合训练侧重算 logprob。
  - Megatron + SGLang：优先看 slime 对齐方案或 ROLL Router Replay。
  - MoE 模型：先开 Router Replay，再做算子/精度对齐。
  - BF16 不稳定：可以试 FP16，但必须同步检查 loss scaling、KV cache 和 softmax 精度。
  - 昇腾栈：可以直接用昇腾提供的训推一致算子集和配套 recipe。
  
如果你告诉我现在用的是哪套组合，比如 verl + vLLM、slime + SGLang、Megatron + vLLM，以及是稠密还是 MoE，我可以帮你直接给一份最小改动清单。